# Kaggle Runner: Pointwise PURS (MovieLens-1M)

This notebook runs your updated binary pointwise PURS pipeline end-to-end on Kaggle.

What it does:
- Sets up repository path in a writable location
- Installs dependencies
- Downloads/checks MovieLens-1M into data/raw/ml-1m
- Runs ExperimentRunner with pointwise settings
- Prints strict public-style PURS metrics: AUC, Hit Rate, Precision, Coverage, Unexpectedness

In [ ]:
from pathlib import Path
import os
import sys
import shutil

candidates = [
    Path('/kaggle/working/recsys'),
    Path('/kaggle/input/recsys'),
    Path('/kaggle/input/recsys/recsys'),
    Path.cwd(),
]

repo_root = None
for candidate in candidates:
    if (candidate / 'config' / 'base_config.py').exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        'Could not locate the recsys repository. Put it in /kaggle/working/recsys or /kaggle/input/recsys.'
    )

if str(repo_root).startswith('/kaggle/input'):
    writable_root = Path('/kaggle/working/recsys')
    if not writable_root.exists():
        shutil.copytree(repo_root, writable_root)
    repo_root = writable_root

os.chdir(repo_root)
if str(repo_root.parent) not in sys.path:
    sys.path.insert(0, str(repo_root.parent))

print('Repository root:', repo_root)
print('Working directory:', Path.cwd())

In [ ]:
import subprocess
import sys
from pathlib import Path

requirements_file = Path('requirements-kaggle.txt')
if not requirements_file.exists():
    requirements_file = Path('requirements.txt')

print('Installing dependencies from', requirements_file)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_file)])
print('Dependency install done.')

In [ ]:
from pathlib import Path
import urllib.request
import zipfile

raw_dir = Path('data/raw')
ml1m_dir = raw_dir / 'ml-1m'
ratings_file = ml1m_dir / 'ratings.dat'

if not ratings_file.exists():
    raw_dir.mkdir(parents=True, exist_ok=True)
    zip_path = raw_dir / 'ml-1m.zip'
    url = 'https://files.grouplens.org/datasets/movielens/ml-1m.zip'
    print('Downloading MovieLens-1M from', url)
    urllib.request.urlretrieve(url, zip_path)
    print('Extracting', zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(raw_dir)

for required in ['ratings.dat', 'movies.dat', 'users.dat']:
    path = ml1m_dir / required
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}')

print('MovieLens-1M is ready at', ml1m_dir)

## Run Mode
- Set `SMOKE_RUN = True` for fast validation
- Set `SMOKE_RUN = False` for full baseline (100 epochs)

In [ ]:
from datetime import datetime

SMOKE_RUN = True

subset = 0.02 if SMOKE_RUN else None
epochs = 1 if SMOKE_RUN else 100
run_tag = 'smoke' if SMOKE_RUN else 'full'
exp_name = f'kaggle_purs_pointwise_{run_tag}_{datetime.now().strftime("%Y%m%d_%H%M%S")}'

print('SMOKE_RUN:', SMOKE_RUN)
print('subset:', subset)
print('epochs:', epochs)
print('experiment:', exp_name)

In [ ]:
from recsys.config import load_config
from recsys.experiments.experiment_runner import ExperimentRunner

config = load_config('kaggle')

# Model/task selection
config.model.model_name = 'purs'
config.data.dataset_name = 'movielens-1m'
config.data.positive_rating_threshold = 3.5

# Baseline hyperparameters
config.model.epochs = epochs
config.model.batch_size = 256
config.model.learning_rate = 0.01
config.model.embedding_dim = 128
config.model.gru_hidden_dim = 128
config.model.dropout = 0.1

# Data/eval settings
config.data.data_subset = subset
config.experiment.k_values = [10]

# Runtime controls
config.experiment.experiment_name = exp_name
config.model.num_workers = 2

runner = ExperimentRunner(config)
results = runner.run()
results

In [ ]:
public_metric_map = {
    'auc': 'AUC',
    'hit_rate': 'Hit Rate (score > 0.5)',
    'precision': 'Precision (same public hit-rate formula)',
    'coverage': 'Coverage',
    'unexpectedness': 'Unexpectedness',
}

print('Public-style PURS metric summary')
print(f"best_selection_metric: {results.get('best_selection_metric', 'auc')}")
if 'best_selection_score' in results:
    print(f"best_selection_score: {results['best_selection_score']:.6f}")

for key, label in public_metric_map.items():
    if key in results:
        print(f"{label} ({key}): {results[key]:.6f}")

print('\nLog directory:', runner.metrics_logger.get_experiment_dir())